# India Map Fourier Animations

This companion notebook independently loads the India outline, computes its Fourier representation, and renders the three animations.

## 1. Animation setup and imports

Configure plotting and animation rendering defaults.

In [ ]:
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle

# Keep embedded JavaScript animations within a practical notebook size.
mpl.rcParams["animation.embed_limit"] = 80

## 2. Load the India outline and compute Fourier data

Load the Natural Earth country geometry, resample India's exterior contour, and verify the arrays before rendering.

In [ ]:
world_url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url)
name_col = "name" if "name" in world.columns else "ADMIN"
india = world[world[name_col] == "India"]
if india.empty:
    raise ValueError("India was not found in the Natural Earth dataset.")

geom = india.geometry.iloc[0]
if geom.geom_type == "MultiPolygon":
    poly = max(geom.geoms, key=lambda geometry: geometry.area)
else:
    poly = geom

coords = np.asarray(poly.exterior.coords)
z_raw = coords[:, 0] + 1j * coords[:, 1]
segment_lengths = np.abs(np.diff(z_raw))
arc = np.concatenate(([0.0], np.cumsum(segment_lengths)))
N = 1024
target_arc = np.linspace(0.0, arc[-1], N, endpoint=False)
x_u = np.interp(target_arc, arc, coords[:, 0])
y_u = np.interp(target_arc, arc, coords[:, 1])
z = x_u + 1j * y_u
C = np.fft.fft(z - z.mean()) / N

if z.ndim != 1 or C.ndim != 1 or len(z) != N or len(C) != N:
    raise ValueError("Expected one-dimensional z and C arrays with length N.")

print(f"Loaded India's outline: {N} contour samples and {len(C)} Fourier coefficients.")

## 3. Fourier reconstruction with increasing harmonics

This animation progressively adds low-frequency Fourier pairs to reconstruct the outline.

In [ ]:
k_frames = np.unique(np.linspace(1, 128, 64, dtype=int))

fig, ax = plt.subplots(figsize=(7.5, 7.5))
base_line, = ax.plot(z.real, z.imag, color="lightgray", linewidth=1.2, label="Original")
recon_line, = ax.plot([], [], color="tab:blue", linewidth=2.0, label="Fourier reconstruction")
ax.set_aspect("equal")
ax.set_xlabel("Real axis (Longitude)")
ax.set_ylabel("Imag axis (Latitude)")
ax.grid(True, alpha=0.25)
ax.legend(loc="best")
title = ax.set_title("Fourier outline movie (K=1)")

pad_x = 0.8
pad_y = 0.8
ax.set_xlim(z.real.min() - pad_x, z.real.max() + pad_x)
ax.set_ylim(z.imag.min() - pad_y, z.imag.max() + pad_y)

def update_reconstruction(frame_idx):
    K = int(k_frames[frame_idx])
    Ck = np.zeros_like(C)
    Ck[0] = C[0]
    Ck[1:K + 1] = C[1:K + 1]
    Ck[-K:] = C[-K:]
    zk = np.fft.ifft(Ck * N) + z.mean()
    recon_line.set_data(zk.real, zk.imag)
    title.set_text(f"Fourier outline movie (K={K})")
    return recon_line, title

ani = FuncAnimation(
    fig, update_reconstruction, frames=len(k_frames), interval=120,
    blit=False, repeat=True,
)
plt.close(fig)
HTML(ani.to_jshtml())

## 4. Trace the reconstructed outline

Draw the reconstructed contour one point at a time.

In [ ]:
K = 160
Ck = np.zeros_like(C)
Ck[0] = C[0]
Ck[1:K + 1] = C[1:K + 1]
Ck[-K:] = C[-K:]
z_rec = np.fft.ifft(Ck * N) + z.mean()

fig, ax = plt.subplots(figsize=(6, 6))
trace_line, = ax.plot([], [], color="tab:blue", linewidth=2.0, label=f"Trace (K={K})")
head_point, = ax.plot([], [], marker="o", markersize=5, color="tab:red")
ax.set_aspect("equal")
ax.set_xlabel("Real axis (Longitude)")
ax.set_ylabel("Imag axis (Latitude)")
ax.set_title("Fourier tracing: one point at a time")
ax.grid(True, alpha=0.25)
ax.legend(loc="best")

pad_x = 0.8
pad_y = 0.8
ax.set_xlim(z.real.min() - pad_x, z.real.max() + pad_x)
ax.set_ylim(z.imag.min() - pad_y, z.imag.max() + pad_y)

# Downsample frames to keep the embedded animation size manageable.
max_frames = 180
frame_indices = np.unique(np.linspace(0, len(z_rec) - 1, max_frames, dtype=int))
n_frames = len(frame_indices)

def init_trace():
    trace_line.set_data([], [])
    head_point.set_data([], [])
    return trace_line, head_point

def update_trace(i):
    idx = int(frame_indices[i])
    trace_line.set_data(z_rec.real[:idx + 1], z_rec.imag[:idx + 1])
    head_point.set_data([z_rec.real[idx]], [z_rec.imag[idx]])
    return trace_line, head_point

ani_trace = FuncAnimation(
    fig, update_trace, init_func=init_trace, frames=n_frames,
    interval=40, blit=True, repeat=True,
)
plt.close(fig)
HTML(ani_trace.to_jshtml())

## 5. Fourier epicycles

Show the rotating Fourier phasors and their traced endpoint.

In [ ]:
# Number of epicycles (largest-amplitude modes first).
K_epi = 60
order = np.argsort(-np.abs(C))
selected = order[:K_epi]
selected_freqs = (np.fft.fftfreq(N) * N)[selected]
selected_C = C[selected]
radii = np.abs(selected_C)

# Precompute the phasor-chain positions for every frame.
n_frames = 300
ts = np.linspace(0, N, n_frames, endpoint=False)
center_offset = z.mean()
all_positions = np.empty((n_frames, K_epi + 1), dtype=complex)
for frame_idx, t in enumerate(ts):
    pos = center_offset
    all_positions[frame_idx, 0] = pos
    for i, (freq, coef) in enumerate(zip(selected_freqs, selected_C)):
        angle = np.angle(coef) + 2 * np.pi * freq * t / N
        pos += np.abs(coef) * np.exp(1j * angle)
        all_positions[frame_idx, i + 1] = pos

tips = all_positions[:, -1]

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect("equal")
pad = 1.5
ax.set_xlim(z.real.min() - pad, z.real.max() + pad)
ax.set_ylim(z.imag.min() - pad, z.imag.max() + pad)
ax.set_title(f"India outline — Fourier epicycles (K={K_epi})")
ax.grid(True, alpha=0.2)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

circles = []
for radius in radii:
    circle = Circle((0, 0), radius, fill=False, color="steelblue", linewidth=0.7, alpha=0.45)
    circle.set_animated(True)
    ax.add_patch(circle)
    circles.append(circle)

arms = [
    ax.plot([], [], color="steelblue", linewidth=0.9, alpha=0.65)[0]
    for _ in range(K_epi)
]
trace_line, = ax.plot([], [], color="tomato", linewidth=1.8, label=f"Trace (K={K_epi})")
head_dot, = ax.plot([], [], "o", color="tomato", markersize=5)
ax.legend(loc="upper right")

def init_epicycles():
    trace_line.set_data([], [])
    head_dot.set_data([], [])
    for arm in arms:
        arm.set_data([], [])
    return circles + arms + [trace_line, head_dot]

def update_epicycles(frame):
    chain = all_positions[frame]
    for i in range(K_epi):
        circles[i].center = (chain[i].real, chain[i].imag)
        arms[i].set_data(
            [chain[i].real, chain[i + 1].real],
            [chain[i].imag, chain[i + 1].imag],
        )
    trace_line.set_data(tips[:frame + 1].real, tips[:frame + 1].imag)
    head_dot.set_data([tips[frame].real], [tips[frame].imag])
    return circles + arms + [trace_line, head_dot]

ani_epi = FuncAnimation(
    fig, update_epicycles, init_func=init_epicycles,
    frames=n_frames, interval=40, blit=True, repeat=True,
)
plt.close(fig)
HTML(ani_epi.to_jshtml())